# Gazette Mistral OCR ETL Prototype

This notebook prototypes the intended ETL flow:

1. Start from PDF URL inputs.
2. Send each PDF URL to Mistral OCR.
3. Consume the Mistral OCR JSON in memory and optionally save it as a raw artifact.
4. Stitch page markdown into one markdown document.
5. Parse the stitched markdown into a JSON envelope.
6. Write the envelope for downstream loading.

Set `MISTRAL_API_KEY` in your environment, paste PDF URLs into `PDF_URL_TEXT`, then run the cells in order.


In [26]:
from __future__ import annotations

import json
import os
import re
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

PROJECT_DIR = Path.cwd()

# API mode is enabled by default. Paste one PDF URL per line below.
RUN_MISTRAL_OCR = True
MISTRAL_MODEL = "mistral-ocr-latest"

# Prefer setting MISTRAL_API_KEY in your environment.
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")

PDF_URL_TEXT = "https://new.kenyalaw.org/akn/ke/officialGazette/2008-07-11/53/eng@2008-07-11/source.pdf"

PDF_URLS = [
    line.strip()
    for line in PDF_URL_TEXT.splitlines()
    if line.strip() and not line.strip().startswith("#")
]

# Optional: a manifest text file containing one PDF URL per line.
MANIFEST_PATH: Path | None = None

# Fallback fixture if you temporarily set RUN_MISTRAL_OCR = False.
SAVED_MISTRAL_JSON = PROJECT_DIR / "gazette_1999-12-03_8000.raw.json"


def _run_name_from_urls(urls: list[str]) -> str:
    if not urls:
        return "mistral_ocr_prototype"
    p = urllib.parse.urlparse(urls[0]).path or "/"
    # Kenyalaw AKN PDFs are often .../source.pdf; name the run after gazette date + number
    # so the output folder matches {RUN_NAME} in {RUN_NAME}.raw.json, etc.
    m = re.search(
        r"/(?:akn/[^/]+/)?officialGazette/(\d{4}-\d{2}-\d{2})/(\d+)/",
        p,
        re.IGNORECASE,
    )
    if m:
        base = f"gazette_{m.group(1)}_{m.group(2)}"
    else:
        base = Path(p).stem or "pdf"
    out = re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_")
    return out or "mistral_ocr_prototype"


RUN_NAME = _run_name_from_urls(PDF_URLS) if RUN_MISTRAL_OCR else SAVED_MISTRAL_JSON.name.removesuffix(".raw.json")
OUT_DIR = PROJECT_DIR / "prototype_outputs" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSON_OUT = OUT_DIR / f"{RUN_NAME}.raw.json"
JOINED_MD_OUT = OUT_DIR / f"{RUN_NAME}_joined.md"
FINAL_JSON_OUT = OUT_DIR / f"{RUN_NAME}_envelope.json"

PROJECT_DIR, OUT_DIR, RUN_NAME, len(PDF_URLS)


(WindowsPath('c:/Users/Ronald Wahome/Documents/gazette_mistral_pipeline'),
 WindowsPath('c:/Users/Ronald Wahome/Documents/gazette_mistral_pipeline/prototype_outputs/gazette_2008-07-11_53'),
 'gazette_2008-07-11_53',
 1)

## 1. Resolve PDF URL Inputs

The declared ETL input is PDF. In this prototype, PDFs are accepted as URLs because the Mistral OCR request uses `document_url`. Paste URLs directly or provide a manifest file containing one PDF URL per line.


In [27]:
def _manifest_entries(path: Path | None) -> list[str]:
    if path is None:
        return []
    lines = path.read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines if line.strip() and not line.strip().startswith("#")]


def resolve_pdf_url_sources(*, pdf_urls: list[str], manifest_path: Path | None) -> list[dict]:
    items: list[dict] = []
    seen: set[str] = set()

    def add_url(value: str):
        url = value.strip()
        if not url:
            return
        if not url.lower().startswith(("http://", "https://")):
            raise ValueError(f"Expected a PDF URL, got: {url}")
        if not urllib.parse.urlparse(url).path.lower().endswith(".pdf"):
            raise ValueError(f"Expected URL path to end with .pdf, got: {url}")
        if url in seen:
            return
        seen.add(url)
        items.append({"kind": "pdf_url", "pdf_url": url})

    for url in pdf_urls:
        add_url(url)

    for url in _manifest_entries(manifest_path):
        add_url(url)

    return items


pdf_items = resolve_pdf_url_sources(pdf_urls=PDF_URLS, manifest_path=MANIFEST_PATH)

print("pdf_items:", len(pdf_items))
pdf_items[:3]


pdf_items: 1


[{'kind': 'pdf_url',
  'pdf_url': 'https://new.kenyalaw.org/akn/ke/officialGazette/2008-07-11/53/eng@2008-07-11/source.pdf'}]

## 2. Mistral OCR

This sends each PDF URL to Mistral and collects one OCR JSON block per PDF. The request body matches the n8n node: `mistral-ocr-latest` with `document.type = document_url`.


In [28]:
MISTRAL_OCR_URL = "https://api.mistral.ai/v1/ocr"


def ocr_pdf_url(pdf_url: str, *, api_key: str, model: str = MISTRAL_MODEL) -> dict:
    body = {
        "model": model,
        "document": {
            "type": "document_url",
            "document_url": pdf_url,
        },
    }
    request = urllib.request.Request(
        MISTRAL_OCR_URL,
        data=json.dumps(body).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        error_body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Mistral OCR failed for {pdf_url}: HTTP {e.code}: {error_body}") from e

    payload.setdefault("pdf_url", pdf_url)
    return payload


def run_mistral_ocr(items: list[dict]) -> list[dict]:
    api_key = MISTRAL_API_KEY.strip() or os.getenv("MISTRAL_API_KEY", "").strip()
    if not api_key:
        raise EnvironmentError(
            "Set MISTRAL_API_KEY in the config cell or as an environment variable before running OCR."
        )

    blocks = []
    for i, item in enumerate(items, start=1):
        print(f"OCR {i}/{len(items)}: {item['pdf_url']}")
        blocks.append(ocr_pdf_url(item["pdf_url"], api_key=api_key))
    return blocks


if RUN_MISTRAL_OCR:
    if not pdf_items:
        raise ValueError("RUN_MISTRAL_OCR is True, but no PDF inputs were configured.")
    mistral_blocks = run_mistral_ocr(pdf_items)
    RAW_JSON_OUT.write_text(json.dumps(mistral_blocks, ensure_ascii=False, indent=2), encoding="utf-8")
    raw_json_path = RAW_JSON_OUT
else:
    raw_json_path = SAVED_MISTRAL_JSON

print("raw_json_path:", raw_json_path)


OCR 1/1: https://new.kenyalaw.org/akn/ke/officialGazette/2008-07-11/53/eng@2008-07-11/source.pdf
raw_json_path: c:\Users\Ronald Wahome\Documents\gazette_mistral_pipeline\prototype_outputs\gazette_2008-07-11_53\gazette_2008-07-11_53.raw.json


## 3. Load and Normalize Mistral OCR JSON

This step flattens one or more Mistral OCR blocks into a single ordered list of pages. Batch JSON arrays are supported.


In [29]:
def load_mistral_blocks(source: Path | dict | list) -> list[dict]:
    if isinstance(source, Path):
        if not source.is_file():
            raise FileNotFoundError(f"No such JSON file: {source}")
        raw = source.read_text(encoding="utf-8")
        if not raw.strip():
            raise ValueError(f"Input JSON is empty: {source}")
        source = json.loads(raw)

    if isinstance(source, dict):
        if "pages" not in source:
            raise ValueError("Mistral OCR JSON object has no 'pages' field.")
        return [source]

    if isinstance(source, list):
        if source and all(isinstance(x, dict) and "pages" in x for x in source):
            return source
        if source and all(isinstance(x, dict) and "markdown" in x for x in source):
            return [{"pages": source}]

    raise ValueError("Unsupported Mistral OCR JSON shape.")


def _page_sort_key(page: dict) -> int:
    try:
        return int(page.get("index"))
    except (TypeError, ValueError):
        return 0


def pages_from_blocks(blocks: list[dict]) -> list[dict]:
    rows = []
    global_index = 0

    for block in blocks:
        for page in sorted(block.get("pages") or [], key=_page_sort_key):
            if not isinstance(page, dict):
                continue
            markdown = page.get("markdown") or ""
            if not markdown.strip():
                continue
            rows.append({
                "index": global_index,
                "original_page_index": _page_sort_key(page),
                "markdown": markdown,
                "pdf_url": block.get("pdf_url"),
                "mistral_doc_id": block.get("id"),
                "model": block.get("model"),
            })
            global_index += 1

    return rows


mistral_blocks = load_mistral_blocks(raw_json_path)
pages = pages_from_blocks(mistral_blocks)

print("mistral_blocks:", len(mistral_blocks))
print("pages:", len(pages))
pages[:1]


mistral_blocks: 1
pages: 64


[{'index': 0,
  'original_page_index': 0,
  'markdown': '![img-0.jpeg](img-0.jpeg)\n\n# THE KENYA GAZETTE\n\nPublished by Authority of the Republic of Kenya\n\n(Registered as a Newspaper at the G.P.O.)\n\nVol. CX—No. 53\n\nNAIROBI, 11th July, 2008\n\nPrice Sh. 50\n\n## CONTENTS\n\n|  GAZETTE NOTICES | PAGE | GAZETTE NOTICES—(Contd.) | PAGE  |\n| --- | --- | --- | --- |\n|  The Restrictive Trade Practices, Monopolies and Price Control Act—Take-over | 1668 | The Standards Act—Declaration Kenya Standards | 1712-1714  |\n|  The Local Authorities Problems Trust Rules—Appointment | 1668 | The Energy Act—Approval of Schedule of Tariffs for Supply of Electricity | 1714-1720  |\n|  The Cotton Act—Notification of Elections, etc. | 1668 | The Kenya Power and Lighting Company Limited—Foreign Exchange Fluctuation Adjustment, etc. | 1721  |\n|  The Exchequer and Audit Act—Appointment | 1668 | The Civil Aviation Act—Corrigenda | 1722  |\n|  The Mutual Tariff Commissions—Common Market for Eastern and 

## 4. Stitch Markdown

This creates one continuous markdown document from the normalized page list.


In [30]:
def join_pages_to_markdown(pages: list[dict], *, add_index_headers: bool = True, add_doc_headers: bool = True) -> str:
    chunks = []
    last_doc_id = object()

    for page in pages:
        doc_id = page.get("mistral_doc_id") or page.get("pdf_url")
        if add_doc_headers and doc_id != last_doc_id:
            title = page.get("pdf_url") or f"mistral_doc_id={page.get('mistral_doc_id')}"
            chunks.append(f"\n\n---\n\n# Document: {title}\n")
            last_doc_id = doc_id

        markdown = page["markdown"].strip()
        if add_index_headers:
            chunks.append(f"\n\n---\n\n## Index {page['index']}\n\n{markdown}\n")
        else:
            chunks.append(markdown)

    return "\n".join(chunks).strip() + "\n"


joined_markdown = join_pages_to_markdown(pages)
JOINED_MD_OUT.write_text(joined_markdown, encoding="utf-8")

print("wrote:", JOINED_MD_OUT)
print("chars:", len(joined_markdown))


wrote: c:\Users\Ronald Wahome\Documents\gazette_mistral_pipeline\prototype_outputs\gazette_2008-07-11_53\gazette_2008-07-11_53_joined.md
chars: 352157


## 5. Parse Markdown to Envelope JSON

This parser is intentionally prototype-level. The regexes and table heuristics can be improved as more gazette layouts are tested.


In [31]:
NOTICE_SPLIT_RE = re.compile(r"(?=(?:GAZETTE|GRZETTE)\s+NOTICE\s+NO\.?\s*\d+)", re.IGNORECASE)
NOTICE_NO_RE = re.compile(r"(?:GAZETTE|GRZETTE)\s+NOTICE\s+NO\.?\s*(\d+)", re.IGNORECASE)
DATE_RE = re.compile(r"\b\d{1,2}(?:st|nd|rd|th)\s+[A-Za-z]+,\s+\d{4}\b")


def split_table_row(line: str) -> list[str]:
    row = line.strip()
    if not row.startswith("|"):
        return []
    row = row[1:-1] if row.endswith("|") else row[1:]
    return [cell.strip() for cell in row.split("|")]


def is_separator_row(line: str) -> bool:
    cells = split_table_row(line)
    return bool(cells) and all(
        not cell.strip() or re.fullmatch(r":?-{3,}:?", cell.strip().replace(" ", ""))
        for cell in cells
    )


def normalize_row(cells: list[str], width: int) -> list[str]:
    cells = list(cells)
    if len(cells) < width:
        cells.extend([""] * (width - len(cells)))
    elif len(cells) > width:
        cells = cells[: width - 1] + [" | ".join(cells[width - 1:])]
    return cells


def extract_markdown_tables(text: str) -> list[dict]:
    lines = text.splitlines()
    tables = []
    i = 0

    while i < len(lines) - 1:
        header_line = lines[i].rstrip()
        sep_line = lines[i + 1].rstrip()

        if not (header_line.lstrip().startswith("|") and is_separator_row(sep_line)):
            i += 1
            continue

        headers = split_table_row(header_line)
        width = max(len(headers), len(split_table_row(sep_line)), 1)
        headers = [h or f"column_{idx}" for idx, h in enumerate(normalize_row(headers, width), start=1)]
        raw_lines = [header_line, sep_line]
        rows = []
        i += 2

        while i < len(lines) and lines[i].lstrip().startswith("|"):
            raw_lines.append(lines[i].rstrip())
            if not is_separator_row(lines[i]):
                row = normalize_row(split_table_row(lines[i]), width)
                if any(cell.strip() for cell in row):
                    rows.append(row)
            i += 1

        tables.append({
            "headers": headers,
            "rows": rows,
            "records": [dict(zip(headers, row)) for row in rows],
            "raw_table_markdown": "\n".join(raw_lines),
        })

    return tables


def markdown_to_envelope(markdown_text: str, *, pages: list[dict], raw_json_path: Path, joined_md_path: Path) -> dict:
    notices = []
    for part in [p.strip() for p in NOTICE_SPLIT_RE.split(markdown_text) if p.strip()]:
        match = NOTICE_NO_RE.search(part)
        if not match:
            continue
        tables = extract_markdown_tables(part)
        notices.append({
            "notice_no": match.group(1),
            "dates_found": DATE_RE.findall(part),
            "table_count": len(tables),
            "tables": tables,
            "raw_markdown": part,
        })

    doc_ids = sorted({str(p.get("mistral_doc_id")) for p in pages if p.get("mistral_doc_id") is not None})
    return {
        "schema_version": "0.1-prototype",
        "pipeline": "pdf -> mistral-ocr-json -> joined-markdown -> envelope-json",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": {
            "raw_mistral_json": str(raw_json_path),
            "joined_markdown": str(joined_md_path),
            "mistral_doc_ids": doc_ids,
        },
        "stats": {
            "document_count": len(doc_ids) or len(mistral_blocks),
            "page_count": len(pages),
            "notice_count": len(notices),
            "char_count_markdown": len(markdown_text),
            "notices_with_tables": sum(1 for notice in notices if notice["table_count"] > 0),
            "total_tables": sum(notice["table_count"] for notice in notices),
        },
        "notices": notices,
    }


final_envelope = markdown_to_envelope(
    joined_markdown,
    pages=pages,
    raw_json_path=raw_json_path,
    joined_md_path=JOINED_MD_OUT,
)
FINAL_JSON_OUT.write_text(json.dumps(final_envelope, ensure_ascii=False, indent=2), encoding="utf-8")

print("wrote:", FINAL_JSON_OUT)
print("stats:", final_envelope["stats"])


wrote: c:\Users\Ronald Wahome\Documents\gazette_mistral_pipeline\prototype_outputs\gazette_2008-07-11_53\gazette_2008-07-11_53_envelope.json
stats: {'document_count': 1, 'page_count': 64, 'notice_count': 204, 'char_count_markdown': 352157, 'notices_with_tables': 13, 'total_tables': 29}


## 6. Quick Preview

Use this cell to sanity-check the envelope before loading it elsewhere.


In [19]:
first_notice = final_envelope["notices"][0] if final_envelope["notices"] else None

if first_notice:
    print("first_notice_no:", first_notice["notice_no"])
    print("dates_found:", first_notice["dates_found"][:5])
    print("table_count:", first_notice["table_count"])
    print(first_notice["raw_markdown"][:1000])
else:
    print("No notices found. Review NOTICE_* regexes or source markdown.")


first_notice_no: 12171
dates_found: []
table_count: 0
Gazette Notice No. 12171 of 2008, amend the expression printed as “District Registrar, Kiambu” to read “District Registrar, Chuka”.

---

##
